# 🔍 Exploratory Data Analysis — Credit Risk Dataset
---
### *Understanding the data before building predictive models*

## 📦 1. Import Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Styling
sns.set_style('whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Warnings
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully!')

## 📂 2. Load Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('credit_risk_dataset.csv')

print(f'✅ Dataset loaded successfully!')
print(f'📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns')

## 👀 3. Quick Glance at the Data

In [ ]:
# First 10 rows
df.head(10)

In [ ]:
# Last 5 rows
df.tail(5)

In [ ]:
# Column names and data types
df_info = pd.DataFrame({
    'Column': df.columns,
    'Data Type': df.dtypes.values,
    'Non-Null Count': df.notnull().sum().values,
    'Null Count': df.isnull().sum().values,
    'Null %': np.round(df.isnull().sum().values / len(df) * 100, 2)
})
df_info

## 📊 4. Statistical Summary

In [ ]:
# Descriptive statistics for numerical columns
df.describe().T.style.background_gradient(cmap='YlOrRd')

In [ ]:
# Summary for categorical columns
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    print(f'\n🔷 {col.upper()}')
    print(f'Unique values: {df[col].nunique()}')
    print(df[col].value_counts().to_string())
    print('-' * 50)

## 🧹 5. Missing Values Analysis

In [ ]:
# Missing values count
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) > 0:
    missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage': (missing/len(df)*100).round(2)})
    print('🔴 Columns with Missing Values:')
    display(missing_df.style.background_gradient(cmap='Reds'))
else:
    print('✅ No missing values found in the dataset!')

In [ ]:
# Visualizing missing values
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=True, cmap='viridis', yticklabels=False)
plt.title('📍 Missing Values Heatmap', fontsize=16, fontweight='bold')
plt.xlabel('Columns')
plt.ylabel('Rows')
plt.tight_layout()
plt.show()
print(f'🟡 Missing values are present in: loan_int_rate and person_emp_length')

## 🔄 6. Duplicate Records

In [ ]:
dup_count = df.duplicated().sum()
print(f'📋 Total duplicate rows: {dup_count}')
if dup_count > 0:
    print(f'🟡 Duplicates account for {round(dup_count/len(df)*100, 2)}% of the data')
else:
    print('✅ No duplicate records found!')

## 📈 7. Distribution Analysis — Numerical Features

In [ ]:
# Numerical columns
num_cols = ['person_age', 'person_income', 'person_emp_length', 
            'loan_amnt', 'loan_int_rate', 'loan_percent_income',
            'cb_person_cred_hist_length']

# Plot distributions
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    if i < len(axes):
        sns.histplot(df[col], kde=True, bins=30, color='royalblue', ax=axes[i])
        axes[i].set_title(f'Distribution of {col}', fontsize=13, fontweight='bold')
        axes[i].set_xlabel('')
        axes[i].axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.2f}')
        axes[i].axvline(df[col].median(), color='green', linestyle='--', label=f'Median: {df[col].median():.2f}')
        axes[i].legend(fontsize=9)

# Hide unused subplots
for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('📊 Distribution of Numerical Features', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 🎯 8. Target Variable Analysis — Loan Status

In [ ]:
# Loan status distribution
status_counts = df['loan_status'].value_counts()
status_pct = df['loan_status'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
ax1 = sns.countplot(x='loan_status', data=df, palette=['#2ecc71', '#e74c3c'], ax=axes[0])
axes[0].set_title('Loan Status Count', fontsize=16, fontweight='bold')
axes[0].set_xlabel('Loan Status (0 = Fully Paid, 1 = Defaulted)')
axes[0].set_ylabel('Count')
for i, p in enumerate(ax1.patches):
    ax1.annotate(f'{int(p.get_height())}\n({status_pct.iloc[i]:.1f}%)', 
                (p.get_x() + p.get_width()/2., p.get_height()), 
                ha='center', va='bottom', fontsize=12, fontweight='bold')

# Pie chart
colors = ['#2ecc71', '#e74c3c']
axes[1].pie(status_counts.values, labels=['Fully Paid (0)', 'Defaulted (1)'], 
            autopct='%1.1f%%', colors=colors, startangle=90, explode=(0.02, 0.02),
            textprops={'fontsize': 12})
axes[1].set_title('Loan Status Proportion', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()
print(f'✅ Class imbalance detected: {status_pct[0]:.1f}% Fully Paid vs {status_pct[1]:.1f}% Defaulted')
print('ℹ️ SMOTE will be used during modeling to handle this imbalance.')

## 🔗 9. Correlation Analysis

In [ ]:
# Correlation matrix
corr_matrix = df.corr(numeric_only=True)

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('🔗 Feature Correlation Matrix', fontsize=18, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Top correlations with loan_status
corr_with_target = corr_matrix['loan_status'].drop('loan_status').sort_values(ascending=False)

plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in corr_with_target.values]
corr_with_target.plot(kind='bar', color=colors)
plt.title('📊 Feature Correlation with Loan Status', fontsize=16, fontweight='bold')
plt.xlabel('Features')
plt.ylabel('Correlation Coefficient')
plt.xticks(rotation=45, ha='right')
plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
plt.tight_layout()
plt.show()

print('🔑 Key findings:')
for col, val in corr_with_target.head(3).items():
    print(f'   • {col}: {val:.3f} (positively correlated with default)')
for col, val in corr_with_target.tail(3).items():
    print(f'   • {col}: {val:.3f} (negatively correlated with default)')

## 📉 10. Bivariate Analysis — Features vs Loan Status

In [ ]:
# Boxplots: Numerical features vs Loan Status
fig, axes = plt.subplots(2, 4, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    if i < len(axes):
        sns.boxplot(x='loan_status', y=col, data=df, 
                   palette=['#2ecc71', '#e74c3c'], ax=axes[i])
        axes[i].set_title(f'{col} vs Loan Status', fontsize=13, fontweight='bold')
        axes[i].set_xlabel('Loan Status')

for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('📊 Feature Distributions by Loan Status', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 🏠 11. Categorical Feature Analysis

In [ ]:
# Categorical columns
cat_plot_cols = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(cat_plot_cols):
    crosstab = pd.crosstab(df[col], df['loan_status'], normalize='index') * 100
    crosstab.plot(kind='bar', stacked=True, color=['#2ecc71', '#e74c3c'], ax=axes[i], width=0.7)
    axes[i].set_title(f'{col} — Default Rate by Category', fontsize=14, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Percentage')
    axes[i].legend(['Fully Paid (0)', 'Defaulted (1)'], loc='upper right')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='right')
    
    # Add percentage labels on bars
    for container in axes[i].containers:
        axes[i].bar_label(container, fmt='%.1f%%', label_type='center', fontsize=9)

plt.suptitle('📊 Categorical Features — Default Rate Analysis', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 🔍 12. Outlier Detection

In [ ]:
# IQR based outlier detection
outlier_cols = ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate']

fig, axes = plt.subplots(1, len(outlier_cols), figsize=(22, 4))

for i, col in enumerate(outlier_cols):
    sns.boxplot(x=df[col], ax=axes[i], color='royalblue', width=0.4)
    axes[i].set_title(f'{col}', fontsize=12, fontweight='bold')
    
    # Calculate IQR
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f'{col}: {len(outliers)} outliers ({round(len(outliers)/len(df)*100, 2)}%)')

plt.suptitle('📍 Outlier Detection — Boxplot Analysis', fontsize=16, fontweight='bold', y=1.1)
plt.tight_layout()
plt.show()

In [ ]:
# Check specific extreme values
print('🔴 Extreme Values:')
print(f'Max person_age: {df["person_age"].max()} (age > 90 will be removed)')
print(f'Max person_emp_length: {df["person_emp_length"].max()}')
print(f'Max loan_amnt: {df["loan_amnt"].max():,}')
print(f'Max person_income: ${df["person_income"].max():,}')

## 🎯 13. Key Insights Summary

In [ ]:
print('''
╔══════════════════════════════════════════════════════════════╗
║              📋 EDA KEY INSIGHTS SUMMARY                    ║
╚══════════════════════════════════════════════════════════════╝

🔴 DATA QUALITY:
   • Missing values found in loan_int_rate and person_emp_length
   • No duplicate records in the dataset
   • Some extreme outliers in age (>90) and employment length

📊 CLASS IMBALANCE:
   • The dataset is imbalanced — fewer defaulted loans than fully paid
   • SMOTE oversampling will be applied during modeling

🔗 TOP CORRELATIONS WITH LOAN STATUS:
   • loan_percent_income — strongest positive correlation
   • loan_int_rate — strong positive correlation
   • loan_grade — correlated with default risk
   • person_income — negative correlation (higher income = lower risk)

🏠 CATEGORICAL INSIGHTS:
   • Loan grade is a strong predictor (A = lowest default, G = highest)
   • Debt consolidation & personal loans have higher default rates
   • Renters have higher default rates than homeowners

📈 DISTRIBUTION NOTES:
   • Most applicants are young (20-35 years old)
   • Income distribution is right-skewed (majority earn < $100K)
   • Common loan amounts are between $5K - $15K

✅ Ready for Feature Engineering! 🚀
''')

print(f'📏 Dataset dimensions: {df.shape}')
print(f'📋 Number of features: {len(df.columns)}')

---
*📌 End of Exploratory Data Analysis. Proceed to **02_feature_engineering.ipynb** for the next step.*